# Retrieval and ranking

## Introduction

Open a film platform, and the home page shows you a few films, picked for you among thousands. How were they picked? There are many ways to recommend films: popular films, films with the same genres, what similar users liked, learned vectors, sequences, graphs... Each one is good at something different. So which one should the platform use?

All of them! But not all at the same time.

Imagine a platform with millions of films. A careful model, with many clues about every film, is far too slow to score millions of films each time a user opens the app. So real systems work in two steps:
- **retrieval**: a few fast recommenders each bring out their best films. Together, they build a short list of a few hundred **candidates**,
- **ranking**: a more careful model sorts the candidates, and we show the first 10.

![Out of all the films, three fast recommenders each bring out 100 films. Together, they build a short list of candidates. A ranker sorts them, and we show the first 10.](images/funnel.svg)

The idea fits in one sentence: *cast a wide net quickly, then sort the catch carefully.*

## Example scenario

We use **MovieLens 100K**, a classic dataset from the [GroupLens research group](https://grouplens.org/datasets/movielens/100k/): 943 users gave 100,000 ratings, from 1 to 5, to 1,682 films. A small `movielens` module downloads it and reads the files. This time, we need the genres too:

In [1]:
from recommendationsystems.movielens import load_movies, load_ratings

_, titles, genres = load_movies()
ratings = load_ratings()
print(ratings[:5])

[[259 254   4]
 [259 285   4]
 [259 297   4]
 [259 184   4]
 [259 172   4]]


Each row is one rating: a user, a film, and a rating from 1 to 5, sorted by time. A rating of 4 or 5 means that the user **liked** the film.

We follow one user through the chapter, user 366. Among the films they liked: Psycho, The Birds, Scream, Evil Dead II, Braindead... A horror fan! Keep them in mind, we will check what each recommender suggests to them.

To know if a recommender is good, we cannot ask the real users. So we **hide the last 10 ratings** of each user, as if they had not happened yet. The recommender only sees the rest, the **history**, and suggests 10 films. Then we reveal the hidden ratings: if at least one of the 10 films was liked later, it is a **hit**!

In [2]:
import numpy as np
from recommendationsystems.movielens import MovieId, UserId

fan = 366
history = {}  # key: user, value: array of (film, rating) the recommender can see
liked_later = {}  # key: user, value: set of hidden films the user liked
for user in np.unique(ratings[:, 0]):
    user_ratings = ratings[ratings[:, 0] == user]
    history[user] = user_ratings[:-10, 1:]
    liked_later[user] = {movie for movie, rating in user_ratings[-10:, 1:] if rating >= 4}

def liked(user: UserId) -> np.ndarray:
    return history[user][history[user][:, 1] >= 4, 0]

Every recommender works the same way: it gives **a score to every film**, and we keep the 10 films with the best score, skipping the films the user has already seen. We test it on every user who liked at least one film before and after the cut. The **hit rate** is the share of these users who got a hit.

In [3]:
from collections.abc import Callable

def top_10(scores: np.ndarray, user: UserId) -> list[MovieId]:
    """Return the 10 best-scored films that the user has not seen yet."""
    seen = set(history[user][:, 0])
    best_first = np.argsort(-scores, kind="stable")  # all films, best score first
    return [int(movie) for movie in best_first if movie not in seen][:10]

test_users = [user for user in history if liked_later[user] and len(liked(user))]

def evaluate(reco: Callable[[UserId], list[MovieId]]) -> float:
    hits = [bool(set(reco(user)) & liked_later[user]) for user in test_users]
    return sum(hits) / len(hits)

Two reference points: random films, and popular films (the films with the most likes, the same list for everybody):

In [4]:
rng = np.random.default_rng(21)

def random_reco(user: UserId) -> list[MovieId]:
    return top_10(rng.random(len(titles)), user)

likes = np.bincount(np.concatenate([liked(user) for user in history]), minlength=len(titles))

def popular_reco(user: UserId) -> list[MovieId]:
    return top_10(likes, user)

print(f"Random: {evaluate(random_reco):.3f}")
print(f"Popular: {evaluate(popular_reco):.3f}")

Random: 0.034
Popular: 0.346


The scores to beat, from the other chapters:
- 0.518 for the **nearest neighbors** (chapter 3): find the 40 users whose likes are closest to yours, and recommend what they liked,
- 0.525 for **SASRec** (chapter 5): a model that reads the last 20 films of a user, in order,
- 0.511 for **LightGCN** (chapter 6): learned vectors that travel along the graph of the likes.

## Retrieval: a short list of candidates

### a. Let's pick four recommenders

For the retrieval, we pick four fast recommenders:
- **popular films**: what everybody likes,
- **genres** (chapter 2): the films whose genres are closest to the genres the user liked,
- **nearest neighbors** (chapter 3): the films liked by the 40 users whose likes are closest to the user's,
- **similar films** (chapter 3): the films liked by the same people as the films the user liked.

All four read the **likes table**: one row per user, one column per film, and a 1 when the user liked the film. We will need them again later, with another history, so the history is an input:

In [5]:
n_users = ratings[:, 0].max() + 1  # user ids start at 1, so row 0 stays empty

def likes_table_of(history: dict) -> np.ndarray:
    table = np.zeros((n_users, len(titles)))
    for user, films in history.items():
        table[user, films[films[:, 1] >= 4, 0]] = 1
    return table

def unit(vectors: np.ndarray) -> np.ndarray:
    """Divide each row by its length, so that a dot product gives a cosine similarity."""
    return vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-9)

Each recommender gives a score to every film, for every user at once: a table with one row per user and one column per film. Three of them compare vectors with the **cosine similarity**: the angle between two vectors, 1 when they point the same way, 0 when they have nothing in common. The `unit` function above turns it into a simple dot product:

In [6]:
def popular_scores(table: np.ndarray) -> np.ndarray:
    return np.tile(table.sum(axis=0), (n_users, 1))  # the same row for everybody

def genre_scores(table: np.ndarray) -> np.ndarray:
    profiles = table @ genres  # the sum of the genres of the liked films
    return unit(profiles) @ unit(genres).T

def neighbor_scores(table: np.ndarray, k: int = 40) -> np.ndarray:
    similarities = unit(table) @ unit(table).T
    np.fill_diagonal(similarities, 0)  # you are not your own neighbor
    far = np.argsort(-similarities, axis=1)[:, k:]
    np.put_along_axis(similarities, far, 0, axis=1)  # only the k nearest neighbors vote
    return similarities @ table

def similar_film_scores(table: np.ndarray) -> np.ndarray:
    film_similarities = unit(table.T) @ unit(table.T).T
    return table @ film_similarities

def score_tables(history: dict) -> dict[str, np.ndarray]:
    table = likes_table_of(history)
    return {
        "popular": popular_scores(table),
        "genres": genre_scores(table),
        "neighbors": neighbor_scores(table),
        "similar films": similar_film_scores(table),
    }

tables = score_tables(history)

### b. Let's gather the candidates

Each recommender brings its 100 best films that the user has not seen. We merge the four lists: a film found twice counts once.

In [7]:
def retrieve(tables: dict, history: dict, user: UserId, n: int = 100) -> np.ndarray:
    seen = set(history[user][:, 0])
    candidates = set()
    for table in tables.values():
        best_first = np.argsort(-table[user], kind="stable")
        candidates.update([int(movie) for movie in best_first if movie not in seen][:n])
    return np.array(sorted(candidates))

fan_candidates = retrieve(tables, history, fan)
print(f"User {fan}: {len(fan_candidates)} candidates out of {len(titles)} films")

User 366: 239 candidates out of 1682 films


Four lists of 100 films, and only 239 candidates: the lists overlap a bit, but each recommender brings films the others missed.

### c. Did we lose the good films?

Retrieval has one job: keep the good films. If a film the user liked later is not among the candidates, no ranker can bring it back.

So let's check: for how many users is at least one liked-later film among the candidates? This is the best hit rate that any ranker could reach, a **ceiling**.

In [8]:
candidates = {user: retrieve(tables, history, user) for user in test_users}

print(f"Candidates per user: {np.mean([len(films) for films in candidates.values()]):.0f} films")
print(f"Ceiling: {np.mean([bool(set(candidates[user]) & liked_later[user]) for user in test_users]):.3f}")

Candidates per user: 246 films
Ceiling: 0.953


With 246 films out of 1,682, one in seven, we keep 95% of the hits. The good films are in the list. Our best recommenders reach about 0.52: the problem is not what we retrieve, it is the order.

## Ranking: the best candidates first

### a. Let's add up the scores

Every candidate now has four scores, one per recommender. The simplest ranking: add them up!

But careful, the scores are not on the same scale. Let's look at the best score of each recommender among the candidates of our fan:

In [9]:
for name, table in tables.items():
    print(f"{name}: {table[fan, fan_candidates].max():.2f}")

popular: 447.00
genres: 0.86
neighbors: 5.14
similar films: 6.00


The popularity counts likes, by the hundreds. A cosine similarity stays below 1. Added as is, the popularity would decide everything.

So, for each user, we divide the scores of each recommender by its best one: the best film of each recommender gets a 1. These numbers are the **clues** of a candidate:

In [10]:
def clues(tables: dict, user: UserId, candidates: np.ndarray) -> np.ndarray:
    """One row per candidate, one column per table: its score, divided by the best one."""
    scores = [table[user, candidates] for table in tables.values()]
    return np.column_stack([score / max(score.max(), 1e-9) for score in scores])

def best_10(candidates: np.ndarray, scores: np.ndarray) -> list[MovieId]:
    return [int(movie) for movie in candidates[np.argsort(-scores, kind="stable")[:10]]]

def sum_reco(user: UserId) -> list[MovieId]:
    return best_10(candidates[user], clues(tables, user, candidates[user]).sum(axis=1))

for movie in sum_reco(fan):
    print(titles[movie])

Shining, The (1980)
Alien (1979)
Star Wars (1977)
Fargo (1996)
Raiders of the Lost Ark (1981)
Aliens (1986)
Return of the Jedi (1983)
Fugitive, The (1993)
Godfather, The (1972)
Empire Strikes Back, The (1980)


### b. How does this recommender perform?

In [11]:
print(f"Sum of the scores: {evaluate(sum_reco):.3f}")

Sum of the scores: 0.513


Worse than the neighbors alone (0.518)! We added more recommenders, and lost a little.

Look at the clues again: the genres count exactly as much as the neighbors. Yet alone, the genres reach a hit rate of 0.119, and the neighbors 0.518. Our weakest recommender weighs as much as our best one.

We could try other weights by hand. Or we could let a model learn them!

## A learned ranker

### a. Where do we find examples?

To learn, the ranker needs examples: candidates, and whether the user liked them later.

We cannot use the hidden ratings: they are our test. Learning from them would be cheating, like reading the answers before the exam.

The trick: let's go one step back in time. We hide the last 10 ratings of each **history** too. This gives an earlier history, and 10 ratings to learn from. Exactly like the test, only 10 ratings earlier:

![The ratings of one user in time order. The test hides the last 10. To train the ranker, we hide 10 more: the ranker learns from these 10, and never sees the test ones.](images/one_step_back.svg)

In [12]:
earlier_history = {}
earlier_liked_later = {}
for user, films in history.items():
    earlier_history[user] = films[:-10]
    earlier_liked_later[user] = {movie for movie, rating in films[-10:] if rating >= 4}

On the earlier history, we do everything again: the score tables, the candidates, the clues. That is why `score_tables` takes the history as input.

Every candidate becomes one example: its clues, and a 1 if the user liked it later, a 0 otherwise.

In [13]:
import torch

def examples(earlier_tables: dict) -> tuple[torch.Tensor, torch.Tensor]:
    inputs, labels = [], []
    for user in earlier_history:
        user_candidates = retrieve(earlier_tables, earlier_history, user)
        inputs.append(clues(earlier_tables, user, user_candidates))
        labels.append(np.isin(user_candidates, list(earlier_liked_later[user])))
    inputs, labels = np.vstack(inputs), np.concatenate(labels)
    return torch.tensor(inputs, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

earlier_tables = score_tables(earlier_history)
inputs, labels = examples(earlier_tables)
print(f"{len(labels):,} examples, {labels.mean():.1%} of them liked later")

226,754 examples, 1.6% of them liked later


### b. Let's train it

Our ranker is the simplest model there is: one weight per clue. The score of a candidate is the weighted sum of its clues, plus a constant:

$$
\text{score} = w_1 \times \text{popular} + w_2 \times \text{genres} + w_3 \times \text{neighbors} + w_4 \times \text{similar films} + b
$$

In PyTorch, this is a single `nn.Linear` layer. For each example, the **sigmoid** turns the score into a probability that the user liked the film. The loss is the **binary cross-entropy**: it is small when the liked films get a probability close to 1, and the others close to 0.

In [14]:
from torch import nn

def train_ranker(inputs: torch.Tensor, labels: torch.Tensor) -> nn.Linear:
    torch.manual_seed(21)
    ranker = nn.Linear(inputs.shape[1], 1)
    optimizer = torch.optim.Adam(ranker.parameters(), lr=0.1)
    for _ in range(2000):  # all the examples at once, 2000 times
        loss = nn.functional.binary_cross_entropy_with_logits(ranker(inputs)[:, 0], labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return ranker

ranker = train_ranker(inputs, labels)
for name, weight in zip(tables, ranker.weight[0].tolist()):
    print(f"{name}: {weight:.2f}")

popular: -0.31
genres: -0.00
neighbors: 1.87
similar films: 2.41


The neighbors and the similar films get big weights. The genres get nothing, and the popularity a little less than nothing. Nobody told the ranker which recommenders were good: it found out from the examples.

### c. Let's recommend

To recommend, the ranker scores the candidates of the real history:

In [15]:
def ranker_reco_of(ranker: nn.Linear, tables: dict) -> Callable[[UserId], list[MovieId]]:
    def reco(user: UserId) -> list[MovieId]:
        with torch.no_grad():
            scores = ranker(torch.tensor(clues(tables, user, candidates[user]), dtype=torch.float32))[:, 0]
        return best_10(candidates[user], scores.numpy())
    return reco

ranker_reco = ranker_reco_of(ranker, tables)
for movie in ranker_reco(fan):
    print(titles[movie])

Shining, The (1980)
Alien (1979)
Aliens (1986)
Fargo (1996)
Star Wars (1977)
Raiders of the Lost Ark (1981)
Terminator, The (1984)
Fugitive, The (1993)
Empire Strikes Back, The (1980)
Usual Suspects, The (1995)


Horror and science fiction at the top, and The Shining first: our fan liked it later, a hit!

### d. How does this recommender perform?

In [16]:
print(f"Learned ranker: {evaluate(ranker_reco):.3f}")

Learned ranker: 0.534


Our best score so far, better than any single recommender! And the ranker never scored all 1,682 films: only the 246 candidates of each user.

## In summary

| Recommender | Uses | Hit rate |
|---|---|---|
| Random | nothing | 0.034 |
| Popular | everybody's likes | 0.346 |
| 40 nearest neighbors (chapter 3) | the most similar users | 0.518 |
| SASRec (chapter 5) | the last 20 films, in order | 0.525 |
| Sum of the scores | 4 recommenders, same weight | 0.513 |
| Learned ranker | 4 recommenders, learned weights | 0.534 |

A real platform does not pick one recommender: it uses several of them, in two steps. **Retrieval** asks a few fast recommenders for their best films, and merges them into a short list of candidates. **Ranking** sorts these candidates with a more careful model, and shows the first ones.

Each step has its own question. For retrieval: did we keep the good films? Here, 246 candidates out of 1,682 films kept 95% of the hits. For ranking: are the good films first? Adding up the scores is not enough: a learned ranker finds by itself how much each recommender should count. And since it only looks at a few hundred films, a real ranker can afford many more clues: the age of the film, the time of day, what the user watched a minute ago...

And that closes our journey. We started in chapter 1 with a single number, the average rating, and ended with a whole pipeline. Along the way, every recommender taught us something: the context of a user, the content of a film, the taste of the neighbors, learned vectors, the order of the films, the graph of the likes. On a real platform, they all have their place, in the net or in the sorting.